In [2]:
#start OMOP infrastucture build for vocab mapping
!pip list
#add needed packages
!pip install sqlalchemy jupyter notebook

Package            Version
------------------ -----------
aiohappyeyeballs   2.6.1
aiohttp            3.13.3
aiosignal          1.4.0
alembic            1.16.5
annotated-doc      0.0.4
anyio              4.12.1
async-timeout      5.0.1
attrs              25.4.0
certifi            2026.1.4
charset-normalizer 3.4.4
click              8.1.8
colorama           0.4.6
colorlog           6.10.1
datasets           4.5.0
dill               0.4.0
evaluate           0.4.6
exceptiongroup     1.3.1
filelock           3.19.1
frozenlist         1.8.0
fsspec             2025.10.0
greenlet           3.2.4
h11                0.16.0
hf-xet             1.2.0
httpcore           1.0.9
httpx              0.28.1
huggingface-hub    1.4.1
idna               3.11
jinja2             3.1.6
joblib             1.5.2
lightgbm           4.6.0
mako               1.3.10
markdown-it-py     3.0.0
markupsafe         3.0.3
mdurl              0.1.2
mpmath             1.3.0
multidict          6.7.1
multiprocess       0.70.18


You should consider upgrading via the 'c:\users\julie\appdata\local\programs\python\python39\python.exe -m pip install --upgrade pip' command.


  Using cached jupyter_console-6.6.3-py3-none-any.whl (24 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl (13 kB)
  Using cached traitlets-5.14.3-py3-none-any.whl (85 kB)
  Using cached jupyter_core-5.8.1-py3-none-any.whl (28 kB)
  Using cached async_lru-2.0.5-py3-none-any.whl (6.1 kB)
  Using cached prompt_toolkit-3.0.52-py3-none-any.whl (391 kB)
  Using cached jupyter_client-8.6.3-py3-none-any.whl (106 kB)
  Using cached comm-0.2.3-py3-none-any.whl (7.3 kB)
  Using cached pandocfilters-1.5.1-py2.py3-none-any.whl (8.7 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
  Using cached jupyterlab_pygments-0.3.0-py3-none-any.whl (15 kB)
  Using cached nbclient-0.10.2-py3-none-any.whl (25 kB)
  Using cached bleach-6.2.0-py3-none-any.whl (163 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl (5.2 kB)
  Using cached jupyter_events-0.12.0-py3-none-any.whl (19 kB)
  Using cached terminado-0.18.1-py3-none-any.whl

You should consider upgrading via the 'c:\users\julie\appdata\local\programs\python\python39\python.exe -m pip install --upgrade pip' command.


In [ ]:
# Create the OMOP CDM Core Tables (Minimal Set)
import sqlite3
from pathlib import Path

DB_PATH = "C:\Users\julie\AI agent courses\dental_imaging_project_local\omop_dental_cbct_v1.db"

def create_omop_tables(db_path = DB_PATH):
    #create minimal OMOP CDM 54. tables for imaging extension demo
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # ============================================================
    # VOCABULARY TABLES (needed to store DICOM concept mappings)
    # ============================================================
    
    cur.execute("""
    CREATE TABLE IF NOT EXISTS concept (
        concept_id          INTEGER PRIMARY KEY,
        concept_name        TEXT NOT NULL,
        domain_id           TEXT NOT NULL,
        vocabulary_id       TEXT NOT NULL,
        concept_class_id    TEXT NOT NULL,
        standard_concept    TEXT,
        concept_code        TEXT NOT NULL,
        valid_start_date    TEXT NOT NULL,
        valid_end_date      TEXT NOT NULL,
        invalid_reason      TEXT
    );
    """)
    
    cur.execute("""
    CREATE TABLE IF NOT EXISTS concept_relationship (
        concept_id_1        INTEGER NOT NULL,
        concept_id_2        INTEGER NOT NULL,
        relationship_id     TEXT NOT NULL,
        valid_start_date    TEXT NOT NULL,
        valid_end_date      TEXT NOT NULL,
        invalid_reason      TEXT
    );
    """)
    
    cur.execute("""
    CREATE TABLE IF NOT EXISTS vocabulary (
        vocabulary_id       TEXT PRIMARY KEY,
        vocabulary_name     TEXT NOT NULL,
        vocabulary_reference TEXT,
        vocabulary_version  TEXT,
        vocabulary_concept_id INTEGER
    );
    """)
    
    # ============================================================
    # CLINICAL DATA TABLES (minimal for demo)
    # ============================================================
    
    cur.execute("""
    CREATE TABLE IF NOT EXISTS person (
        person_id                   INTEGER PRIMARY KEY,
        gender_concept_id           INTEGER NOT NULL,
        year_of_birth               INTEGER NOT NULL,
        month_of_birth              INTEGER,
        day_of_birth                INTEGER,
        birth_datetime              TEXT,
        race_concept_id             INTEGER NOT NULL,
        ethnicity_concept_id        INTEGER NOT NULL,
        location_id                 INTEGER,
        provider_id                 INTEGER,
        care_site_id                INTEGER,
        person_source_value         TEXT,
        gender_source_value         TEXT,
        gender_source_concept_id    INTEGER,
        race_source_value           TEXT,
        race_source_concept_id      INTEGER,
        ethnicity_source_value      TEXT,
        ethnicity_source_concept_id INTEGER
    );
    """)

    # ============================================================
    # IMAGING EXTENSION TABLES (from Park et al. / DICOM2OMOP)
    # ============================================================
    cur.execute("""
    CREATE TABLE IF NOT EXISTS image_occurence (
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS image_feature(
    );
    """)

    conn.commit()
    print(f"OMOP tables created in {db_path}")

    #verify working - sqlite_master special hidden table in every SQLiite db, return names of tables in db
    cur.execute("SELECT name from sqlite_master WHERE type='table';")   #list of tuples, need first
    tables = [row[0] for row in cur.fetchall()]
    print(f"Tables: {', '.join(tables)}")

    conn.close()
    return db_path

if __name__ == '__main__':
    create_omop_tables()
 